# D. Ekstraksi Fitur Menggunakan TF-IDF Vectorization

In [ ]:
# ==============================
# 📦 1. Import Library
# ==============================
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse

# ==============================
# 📂 2. Load Dataset
# ==============================
df = pd.read_csv("dataset_ready_for_feature_extraction.csv")

# Ambil kolom teks dan label
X_text = df["teks_final"].astype(str)
y = df["label_num"]

# ==============================
# 🔠 3. TF-IDF Vectorization
# ==============================
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,     # ambil 5000 kata paling penting
    sublinear_tf=True,     # normalisasi logaritmik TF
    encoding='utf-8',
)

X_tfidf = tfidf_vectorizer.fit_transform(X_text)

# ==============================
# 📊 4. Info Bentuk Matriks
# ==============================
print("✅ TF-IDF selesai diekstraksi.")
print("📏 Bentuk matriks TF-IDF:", X_tfidf.shape)

# ==============================
# 🔍 5. Contoh Nilai TF-IDF (non-zero)
# ==============================
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(X_tfidf[:5].toarray(), columns=feature_names)

for i in range(5):
    print(f"\n📰 Dokumen {i+1}:")
    nonzero = tfidf_df.loc[i][tfidf_df.loc[i] > 0]
    print(nonzero.sort_values(ascending=False).head(10))  # tampilkan 10 kata terpenting per dokumen

# ==============================
# 💾 6. Simpan hasil ekstraksi TF-IDF
# ==============================
scipy.sparse.save_npz("tfidf_features.npz", X_tfidf)

print("\n💾 Hasil ekstraksi TF-IDF berhasil disimpan!")
print("📂 File fitur: tfidf_features.npz")
print("📏 Bentuk fitur:", X_tfidf.shape)


✅ TF-IDF selesai diekstraksi.
📏 Bentuk matriks TF-IDF: (6760, 5000)

📰 Dokumen 1:
anies       0.305029
pilkada     0.264565
gubernur    0.243599
tirto       0.221707
rano        0.218156
baswedan    0.213974
calon       0.201270
urut        0.191479
batik       0.185898
mas         0.164271
Name: 0, dtype: float64

📰 Dokumen 2:
aparatur      0.322623
prabowo       0.246483
ancam         0.243710
rakyat        0.239616
fabricated    0.220051
urus          0.210558
pimpin        0.208759
presiden      0.187179
valid         0.184254
negara        0.152427
Name: 1, dtype: float64

📰 Dokumen 3:
menkes        0.313770
pandemic      0.271661
budi          0.254752
pandemi       0.246330
order         0.230614
new           0.175992
agenda        0.171384
gagas         0.158006
world         0.155813
misleading    0.136996
Name: 2, dtype: float64

📰 Dokumen 4:
militer       0.308229
wajib         0.245363
putri         0.213269
prabowo       0.211409
putra         0.204693
rakyat        0.197

**Contoh Representasi TF-IDF (10 Fitur Teratas)**

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=10)  # ambil 10 kata teratas agar tabel tidak terlalu besar
X_tfidf = tfidf_vectorizer.fit_transform(X_text)

# 4️⃣ Buat DataFrame hasil ekstraksi
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
).round(3)

# 5️⃣ Ambil hanya baris yang punya nilai TF-IDF ≠ 0.0
tfidf_nonzero = tfidf_df[(tfidf_df != 0).any(axis=1)]

# 6️⃣ Tampilkan hanya beberapa baris agar tabel ringkas
print("📊 Contoh representasi hasil ekstraksi TF-IDF (hanya bobot ≠ 0.0):")
print(tfidf_nonzero.head(10))


📊 Contoh representasi hasil ekstraksi TF-IDF (hanya bobot ≠ 0.0):
    akun  content  fakta  hasil  indonesia  konten  periksa   temu  unggah  \
0  0.481    0.228  0.394  0.113      0.143   0.405    0.383  0.000   0.467   
1  0.192    0.273  0.158  0.135      0.000   0.486    0.153  0.167   0.747   
2  0.429    0.305  0.352  0.000      0.191   0.543    0.171  0.187   0.417   
3  0.362    0.257  0.148  0.510      0.323   0.305    0.144  0.158   0.527   
4  0.393    0.279  0.322  0.138      0.351   0.497    0.313  0.171   0.382   
5  0.900    0.116  0.134  0.000      0.000   0.207    0.261  0.071   0.159   
6  0.332    0.472  0.272  0.000      0.000   0.560    0.529  0.000   0.000   
7  0.553    0.157  0.091  0.545      0.099   0.186    0.176  0.000   0.537   
8  0.411    0.292  0.337  0.289      0.183   0.346    0.327  0.358   0.399   
9  0.639    0.227  0.131  0.000      0.000   0.269    0.255  0.000   0.621   

   video  
0  0.000  
1  0.000  
2  0.174  
3  0.000  
4  0.000  
5  0.133 

# E. Ekstraksi Fitur Word2Vec: Pelatihan Model & Representasi Dokumen

In [ ]:
pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 69.3 MB/s eta 0:00:00


In [ ]:
# ================================================================
# 📦 1. Import Library
# ================================================================
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
import joblib

# ================================================================
# 📂 2. Load Dataset
# ================================================================
df = pd.read_csv("dataset_ready_for_feature_extraction.csv")

# Ambil kolom teks saja
X_text = df["teks_final"].astype(str)

# ================================================================
# 🧠 3. Word2Vec Feature Extraction
# ================================================================
print("🔹 Ekstraksi fitur Word2Vec dimulai...")

# 3.1 Tokenisasi sederhana (ubah teks jadi list kata)
sentences = [text.split() for text in X_text]

# 3.2 Latih model Word2Vec
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,   # dimensi vektor tiap kata
    window=5,          # konteks kata (5 kata di sekitar)
    min_count=2,       # hanya kata muncul ≥2 kali yang dilatih
    workers=4,
    sg=1               # gunakan skip-gram (lebih baik untuk data kecil)
)

# Simpan model Word2Vec agar bisa digunakan lagi nanti
w2v_model.save("word2vec_model.model")

print("✅ Model Word2Vec selesai dilatih.")
print("📏 Jumlah kata dalam kosakata model:", len(w2v_model.wv.index_to_key))

# ================================================================
# 🧩 4. Representasi Dokumen
# ================================================================
# Fungsi untuk menghitung rata-rata vektor dari kata-kata dalam 1 dokumen
def document_vector(doc):
    words = [word for word in doc if word in w2v_model.wv.index_to_key]
    if len(words) == 0:
        return np.zeros(w2v_model.vector_size)
    return np.mean(w2v_model.wv[words], axis=0)

# 4.1 Ubah semua dokumen menjadi vektor
X_w2v = np.array([document_vector(doc) for doc in sentences])

print("✅ Dokumen berhasil diubah menjadi vektor.")
print("📏 Bentuk matriks Word2Vec:", X_w2v.shape)

# ================================================================
# 💾 5. Simpan Hasil Ekstraksi
# ================================================================
joblib.dump(X_w2v, "word2vec_features.joblib")

print("\n💾 Fitur Word2Vec disimpan sebagai 'word2vec_features.joblib'")
print("Model Word2Vec disimpan sebagai 'word2vec_model.model'")


🔹 Ekstraksi fitur Word2Vec dimulai...
✅ Model Word2Vec selesai dilatih.
📏 Jumlah kata dalam kosakata model: 23891
✅ Dokumen berhasil diubah menjadi vektor.
📏 Bentuk matriks Word2Vec: (6760, 100)

💾 Fitur Word2Vec disimpan sebagai 'word2vec_features.joblib'
Model Word2Vec disimpan sebagai 'word2vec_model.model'


# Train–Test Split untuk Fitur TF-IDF

In [ ]:
# ==============================
# 📦 1. Import Library
# ==============================
from sklearn.model_selection import train_test_split
import joblib
import scipy.sparse
import pandas as pd


In [ ]:
# ==============================
# 📂 2. Load Data
# ==============================
# Baca dataset untuk mengambil label
df = pd.read_csv("dataset_ready_for_feature_extraction.csv")
y = df["label_num"]

# 🔹 Load TF-IDF
X_tfidf = scipy.sparse.load_npz("tfidf_features.npz")

# 🔹 Load Word2Vec
X_w2v = joblib.load("word2vec_features.joblib")


In [ ]:
# ==============================
# 📂 2. Load Data
# ==============================
# Baca dataset untuk mengambil label
df = pd.read_csv("dataset_ready_for_feature_extraction.csv")
y = df["label_num"]

# 🔹 Load TF-IDF
X_tfidf = scipy.sparse.load_npz("tfidf_features.npz")

# 🔹 Load Word2Vec
X_w2v = joblib.load("word2vec_features.joblib")


# ==============================
# ✂️ 3. Split Dataset (TF-IDF)
# ==============================
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(
    X_tfidf, y,
    test_size=0.2,         # 80% train, 20% test
    random_state=42,       # untuk hasil yang konsisten
    stratify=y             # agar proporsi label tetap sama
)

print("✅ TF-IDF berhasil di-split.")
print("📏 Train:", X_train_tfidf.shape, " Test:", X_test_tfidf.shape)
print("🔸 Distribusi label Train:\n", y_train_tfidf.value_counts(normalize=True))
print("🔹 Distribusi label Test:\n", y_test_tfidf.value_counts(normalize=True))


✅ TF-IDF berhasil di-split.
📏 Train: (5408, 5000)  Test: (1352, 5000)
🔸 Distribusi label Train:
 label_num
0    0.575629
1    0.424371
Name: proportion, dtype: float64
🔹 Distribusi label Test:
 label_num
0    0.575444
1    0.424556
Name: proportion, dtype: float64


# Train–Test Split untuk Fitur Word2Vec

In [ ]:
# ==============================
# ✂️ 4. Split Dataset (Word2Vec)
# ==============================
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n✅ Word2Vec berhasil di-split.")
print("📏 Train:", X_train_w2v.shape, " Test:", X_test_w2v.shape)
print("🔸 Distribusi label Train:\n", y_train_w2v.value_counts(normalize=True))
print("🔹 Distribusi label Test:\n", y_test_w2v.value_counts(normalize=True))



✅ Word2Vec berhasil di-split.
📏 Train: (5408, 100)  Test: (1352, 100)
🔸 Distribusi label Train:
 label_num
0    0.575629
1    0.424371
Name: proportion, dtype: float64
🔹 Distribusi label Test:
 label_num
0    0.575444
1    0.424556
Name: proportion, dtype: float64


**Menyimpan hasil split Dataset**

In [ ]:
# ==============================
# 💾 5. Simpan Hasil Split
# ==============================
joblib.dump((X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf), "split_tfidf.joblib")
joblib.dump((X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v), "split_word2vec.joblib")

print("\n💾 Split dataset berhasil disimpan!")



💾 Split dataset berhasil disimpan!


# **F. Balancing Data Menggunakan SMOTE**

In [ ]:
# ==============================
# 📦 1. Import Library
# ==============================
from imblearn.over_sampling import SMOTE
import joblib
import pandas as pd

# ==============================
# 📂 2. Load Hasil Split Dataset
# ==============================
# Ambil hasil split dari tahap sebelumnya (TF-IDF dan Word2Vec)
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = joblib.load("split_tfidf.joblib")
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = joblib.load("split_word2vec.joblib")

# ==============================
# ⚖️ 3. Balancing Data (SMOTE)
# ==============================
print("🔹 Proses SMOTE dimulai...")

# Inisialisasi SMOTE
smote = SMOTE(random_state=42, sampling_strategy='auto')

# --- TF-IDF ---
X_train_tfidf_bal, y_train_tfidf_bal = smote.fit_resample(X_train_tfidf, y_train_tfidf)
print("\n✅ SMOTE (TF-IDF) selesai.")
print("📏 Sebelum SMOTE:", X_train_tfidf.shape, " | Sesudah SMOTE:", X_train_tfidf_bal.shape)
print("📊 Distribusi label sesudah SMOTE (TF-IDF):")
print(pd.Series(y_train_tfidf_bal).value_counts())

# --- Word2Vec ---
X_train_w2v_bal, y_train_w2v_bal = smote.fit_resample(X_train_w2v, y_train_w2v)
print("\n✅ SMOTE (Word2Vec) selesai.")
print("📏 Sebelum SMOTE:", X_train_w2v.shape, " | Sesudah SMOTE:", X_train_w2v_bal.shape)
print("📊 Distribusi label sesudah SMOTE (Word2Vec):")
print(pd.Series(y_train_w2v_bal).value_counts())

# ==============================
# 💾 4. Simpan Hasil SMOTE
# ==============================
joblib.dump((X_train_tfidf_bal, y_train_tfidf_bal, X_test_tfidf, y_test_tfidf), "balanced_tfidf.joblib")
joblib.dump((X_train_w2v_bal, y_train_w2v_bal, X_test_w2v, y_test_w2v), "balanced_word2vec.joblib")

print("\n💾 Hasil balancing (SMOTE) berhasil disimpan!")
print("📂 File: balanced_tfidf.joblib & balanced_word2vec.joblib")


🔹 Proses SMOTE dimulai...

✅ SMOTE (TF-IDF) selesai.
📏 Sebelum SMOTE: (5408, 5000)  | Sesudah SMOTE: (6226, 5000)
📊 Distribusi label sesudah SMOTE (TF-IDF):
label_num
0    3113
1    3113
Name: count, dtype: int64

✅ SMOTE (Word2Vec) selesai.
📏 Sebelum SMOTE: (5408, 100)  | Sesudah SMOTE: (6226, 100)
📊 Distribusi label sesudah SMOTE (Word2Vec):
label_num
0    3113
1    3113
Name: count, dtype: int64

💾 Hasil balancing (SMOTE) berhasil disimpan!
📂 File: balanced_tfidf.joblib & balanced_word2vec.joblib


# G. Seleksi Fitur (Chi-Square, ANOVA F-test, Mutual Information)

In [ ]:
  # ================================================================
  # 📦 1. Import Library
  # ================================================================
  import warnings
  warnings.filterwarnings("ignore", category=UserWarning)
  warnings.filterwarnings("ignore", category=RuntimeWarning)

  import joblib
  import numpy as np
  import pandas as pd
  from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif

  # ================================================================
  # 📂 2. Load Data Hasil SMOTE (TF-IDF)
  # ================================================================
  data_tfidf = joblib.load("balanced_tfidf.joblib")

  # Unpack sesuai urutan (train_X, train_y, test_X, test_y)
  X_train_tfidf, y_train_tfidf, X_test_tfidf, y_test_tfidf = data_tfidf

  print("✅ Data TF-IDF hasil SMOTE berhasil dimuat.")
  print("📏 Bentuk fitur (train):", X_train_tfidf.shape)
  print("📊 Distribusi label:")
  print(y_train_tfidf.value_counts())

  # Pastikan label dalam bentuk array numerik
  y_train_tfidf = np.array(y_train_tfidf, dtype=int)

  # ================================================================
  # ⚙️ 3. Seleksi Fitur (3 Metode)
  # ================================================================
  K = 2000  # jumlah fitur terpilih

  # -----------------------------
  # 3.1 Chi-Square
  # -----------------------------
  chi2_selector = SelectKBest(score_func=chi2, k=K)
  X_train_chi2 = chi2_selector.fit_transform(X_train_tfidf, y_train_tfidf)
  X_test_chi2 = chi2_selector.transform(X_test_tfidf)

  chi2_scores = chi2_selector.scores_
  chi2_mean = np.nanmean(chi2_scores)
  chi2_top_idx = np.argsort(chi2_scores)[-10:][::-1]

  print("\n🔹 Chi-Square")
  print(f"   ➜ {X_train_tfidf.shape} → {X_train_chi2.shape}")
  print(f"   📈 Rata-rata skor fitur: {chi2_mean:.4f}")
  print(f"   🏆 Top 10 fitur terbaik (index): {chi2_top_idx.tolist()}")

  # -----------------------------
  # 3.2 ANOVA F-test
  # -----------------------------
  anova_selector = SelectKBest(score_func=f_classif, k=K)
  X_train_anova = anova_selector.fit_transform(X_train_tfidf, y_train_tfidf)
  X_test_anova = anova_selector.transform(X_test_tfidf)

  anova_scores = anova_selector.scores_
  anova_mean = np.nanmean(anova_scores)
  anova_top_idx = np.argsort(anova_scores)[-10:][::-1]

  print("\n🔹 ANOVA F-test")
  print(f"   ➜ {X_train_tfidf.shape} → {X_train_anova.shape}")
  print(f"   📈 Rata-rata skor fitur: {anova_mean:.4f}")
  print(f"   🏆 Top 10 fitur terbaik (index): {anova_top_idx.tolist()}")

  # -----------------------------
  # 3.3 Mutual Information
  # -----------------------------
  mi_selector = SelectKBest(score_func=mutual_info_classif, k=K)
  X_train_mi = mi_selector.fit_transform(X_train_tfidf, y_train_tfidf)
  X_test_mi = mi_selector.transform(X_test_tfidf)

  mi_scores = mi_selector.scores_
  mi_mean = np.nanmean(mi_scores)
  mi_top_idx = np.argsort(mi_scores)[-10:][::-1]

  print("\n🔹 Mutual Information")
  print(f"   ➜ {X_train_tfidf.shape} → {X_train_mi.shape}")
  print(f"   📈 Rata-rata skor fitur: {mi_mean:.4f}")
  print(f"   🏆 Top 10 fitur terbaik (index): {mi_top_idx.tolist()}")

  # ================================================================
  # 💾 4. Simpan Hasil Seleksi Fitur
  # ================================================================
  joblib.dump((X_train_chi2, X_test_chi2, y_train_tfidf, y_test_tfidf), "selected_features_chi2.joblib")
  joblib.dump((X_train_anova, X_test_anova, y_train_tfidf, y_test_tfidf), "selected_features_anova.joblib")
  joblib.dump((X_train_mi, X_test_mi, y_train_tfidf, y_test_tfidf), "selected_features_mi.joblib")

  print("\n💾 Hasil seleksi fitur berhasil disimpan!")
  print("📂 File:")
  print(" - selected_features_chi2.joblib")
  print(" - selected_features_anova.joblib")
  print(" - selected_features_mi.joblib")

  # ================================================================
  # 🧾 5. Ringkasan Perbandingan
  # ================================================================
  comparison = pd.DataFrame({
      "Metode": ["Chi-Square", "ANOVA F-test", "Mutual Information"],
      "Fitur Awal": [X_train_tfidf.shape[1]] * 3,
      "Fitur Terpilih": [K] * 3,
      "Rata-rata Skor": [chi2_mean, anova_mean, mi_mean]
  })

  print("\n📊 Perbandingan Hasil Seleksi Fitur:")
  print(comparison.to_string(index=False))


✅ Data TF-IDF hasil SMOTE berhasil dimuat.
📏 Bentuk fitur (train): (6226, 5000)
📊 Distribusi label:
label_num
0    3113
1    3113
Name: count, dtype: int64

🔹 Chi-Square
   ➜ (6226, 5000) → (6226, 2000)
   📈 Rata-rata skor fitur: 3.4665
   🏆 Top 10 fitur terbaik (index): [3580, 3725, 3830, 1527, 2259, 1199, 4766, 106, 3666, 2024]

🔹 ANOVA F-test
   ➜ (6226, 5000) → (6226, 2000)
   📈 Rata-rata skor fitur: 62.9728
   🏆 Top 10 fitur terbaik (index): [3580, 1527, 3725, 3830, 3666, 1199, 2259, 1850, 2024, 808]

🔹 Mutual Information
   ➜ (6226, 5000) → (6226, 2000)
   📈 Rata-rata skor fitur: 0.0125
   🏆 Top 10 fitur terbaik (index): [1199, 3666, 2024, 3308, 1850, 2259, 4291, 1517, 806, 4583]

💾 Hasil seleksi fitur berhasil disimpan!
📂 File:
 - selected_features_chi2.joblib
 - selected_features_anova.joblib
 - selected_features_mi.joblib

📊 Perbandingan Hasil Seleksi Fitur:
            Metode  Fitur Awal  Fitur Terpilih  Rata-rata Skor
        Chi-Square        5000            2000        3.4